# PySpark 3: types, functions and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-03-types-functions-and-nulls.ipynb)

A vendor file has amounts like `"1,204.50"`, dates like `"03/01/2026"`, missing regions, emails in mixed case,
and one amount of `"N/A"`. Before any join or report, every value needs the right type and a deliberate
answer for "what if it is missing or wrong?".

This lesson covers Spark's data types, casting under Spark 4's ANSI mode, how nulls behave, conditional logic,
and the built-in functions from `pyspark.sql.functions` you will use in almost every pipeline: columns,
sorting, strings, numbers, dates, aggregates, hashing, arrays, maps, JSON and UDFs. The
[function map](#map-of-the-functions-in-this-lesson) lists all of them with the section that runs each one.
Window functions get a short preview here and a page of their own,
[PySpark window functions](pyspark-window-functions.ipynb).

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

For everyday transformation work, focus first on column expressions, safe casts, null handling, and `when`. Use the string and date sections when turning raw source fields into join keys and business dates. Each example runs on a small input so you can predict what happens to a missing or invalid value before running the cell.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F
from decimal import Decimal

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Map of the functions in this lesson

Every example calls functions as `F.<name>` after `from pyspark.sql import functions as F`. The table groups the
functions most pipelines use and links to the section that explains and runs them.

| group | functions | section |
| --- | --- | --- |
| columns | `col`, `column`, `lit`, `expr`, `broadcast` | [Read a Spark column expression](#read-a-spark-column-expression) |
| conditionals and nulls | `when` / `otherwise`, `coalesce`, `isnull`, `isnotnull`, `isnan`, `nvl`, `nvl2`, `ifnull`, `nullif`, `nanvl` | [Handle nulls](#handle-nulls), [Write conditional logic](#write-conditional-logic) |
| sorting | `asc`, `desc`, `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last` | [Sort with nulls first or last](#sort-with-nulls-first-or-last) |
| strings | `concat`, `concat_ws`, `upper`, `lower`, `initcap`, `trim`, `ltrim`, `rtrim`, `substring`, `left`, `right`, `split`, `split_part`, `regexp_replace`, `regexp_extract`, `length`, `lpad`, `rpad`, `like`, `rlike`, `contains`, `startswith`, `endswith`, `format_string`, `format_number`, `translate`, `replace` | [Work with strings](#work-with-strings) |
| math | `round`, `bround`, `ceil`, `floor`, `abs`, `sqrt`, `pow`, `log`, `exp`, `greatest`, `least`, `rand`, `try_divide`, `try_add`, `try_multiply` | [Round and compute numbers](#round-and-compute-numbers), [Cast types and handle bad values](#cast-types-and-handle-bad-values) |
| dates and timestamps | `current_date`, `current_timestamp`, `to_date`, `to_timestamp`, `date_format`, `year`, `month`, `dayofmonth`, `dayofweek`, `hour`, `minute`, `quarter`, `weekofyear`, `date_add`, `date_sub`, `add_months`, `datediff`, `months_between`, `date_trunc`, `trunc`, `last_day`, `next_day`, `unix_timestamp`, `from_unixtime`, `from_utc_timestamp`, `to_utc_timestamp`, `window` | [Work with dates and timestamps](#work-with-dates-and-timestamps) |
| aggregates | `count`, `count_distinct`, `approx_count_distinct`, `sum`, `avg` / `mean`, `min`, `max`, `collect_list`, `collect_set`, `first`, `last`, `max_by`, `min_by`, `stddev`, `variance`, `corr`, `percentile_approx`, `median`, `count_if`, `sum_distinct` | [Aggregate functions](#aggregate-functions) |
| window functions | `row_number`, `rank`, `dense_rank`, `lag`, `lead`, `ntile`, `percent_rank`, `cume_dist` | [Window functions: a preview](#window-functions-a-preview), then [PySpark window functions](pyspark-window-functions.ipynb) |
| hashing and ids | `md5`, `sha2`, `hash`, `xxhash64`, `monotonically_increasing_id`, `uuid` | [Generate ids and hashes](#generate-ids-and-hashes) |
| arrays, maps and structs | `explode`, `explode_outer`, `posexplode`, `array`, `array_contains`, `array_distinct`, `size`, `array_join`, `sort_array`, `flatten`, `arrays_zip`, `element_at`, `transform`, `filter`, `aggregate`, `exists`, `struct`, `create_map`, `map_keys`, `map_values` | [Use small arrays and maps in column expressions](#use-small-arrays-and-maps-in-column-expressions) |
| JSON | `from_json`, `to_json`, `get_json_object`, `json_tuple`, `schema_of_json` | [Parse and build JSON](#parse-and-build-json) |
| misc and UDFs | `input_file_name`, `spark_partition_id`, `udf`, `pandas_udf` | [File lineage, partitions and UDFs](#file-lineage-partitions-and-udfs) |

Spark 4 has about 700 functions; the rest are in the
[`pyspark.sql.functions` reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html).

## Read a Spark column expression

Start with the built-in expressions you'll see throughout the lesson. `F.col("status")` refers to a field in the current row. `F.lit("paid")` supplies the same constant to every row. Functions such as `F.lower("status")` accept a column name as shorthand. In `F.coalesce("region", F.lit("unknown"))`, the first string is a column name and the second is a literal fallback.

Use `select` to choose and name the output columns. Use `withColumn` to add or replace a column while keeping the others. Both return a new DataFrame. Spark builds the execution plan as you chain expressions and evaluates it when you call an action such as `show()`. Keeping the logic in built-in column expressions gives Spark a plan it can optimize.

| Need | Column expression |
| --- | --- |
| Name a calculated output | `(F.col("amount") * 2).alias("double_amount")` |
| Use a Spark SQL expression | `F.expr("amount * 2")` returns a column you can pass to `select` |
| Test a closed range | `F.col("amount").between(100, 200)` includes both endpoints |
| Sort with an explicit missing-value policy | `F.col("amount").desc_nulls_last()` puts highest amounts first and nulls last |
| Make an unknown boolean an explicit false | `F.coalesce(F.col("amount") > 100, F.lit(False))` |
| Compare two nullable columns | `F.col("old").eqNullSafe(F.col("new"))` treats two nulls as equal |

See the [PySpark Column API](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/column.html) and [built-in functions by category](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/functions.html).

### Test a range, compare snapshots, and sort nulls

Two order amounts sit at the range endpoints, and one is missing. Compare the current and previous amounts.

In [ ]:
amounts = spark.createDataFrame(
    [(1, 100, 100), (2, 200, None), (3, None, None)],
    "id int, amount int, previous int",
)
amount_expr = F.col("amount")
amounts.select(
    "id", "amount",
    (amount_expr * F.lit(2)).alias("doubled"),
    F.expr("amount * 2").alias("sql_doubled"),
    amount_expr.between(100, 200).alias("in_range"),
    F.coalesce(amount_expr > 100, F.lit(False)).alias("over_100"),
    amount_expr.eqNullSafe(F.col("previous")).alias("unchanged"),
).orderBy(amount_expr.desc_nulls_last()).show()

**Expected output**

```text
+---+------+-------+-----------+--------+--------+---------+
| id|amount|doubled|sql_doubled|in_range|over_100|unchanged|
+---+------+-------+-----------+--------+--------+---------+
|  2|   200|    400|        400|    true|    true|    false|
|  1|   100|    200|        200|    true|   false|     true|
|  3|  NULL|   NULL|       NULL|    NULL|   false|     true|
+---+------+-------+-----------+--------+--------+---------+
```

**What to notice:** Both endpoints are in range. A missing amount gives a null range test, sorts last, and compares equal to a missing previous amount.

### Refer to a column, and broadcast a small table

`F.column("status")` is an alias of `F.col`; both return a `Column` you can do arithmetic on and call methods
such as `.cast`, `.alias` or `.isNull` on. A plain string works only where a function accepts a column name:
`"amount" * 2` is a Python error, `F.col("amount") * 2` is a column expression.

`F.broadcast(df)` marks a small DataFrame to be copied to every executor for a join, so the large side does not
have to be shuffled. Use it for lookup tables (countries, products, statuses) that fit comfortably in memory.
[Lesson 6](pyspark-06-joins.ipynb) covers join strategies in depth.

In [ ]:
import contextlib
import io

orders.select(
    F.column("order_id"),
    (F.column("amount") * F.lit(D("1.2"))).alias("gross"),
    F.lit("EUR").alias("currency"),
).show(3)

joined = orders.join(F.broadcast(customers), "customer_id")
plan = io.StringIO()
with contextlib.redirect_stdout(plan):  # capture the text explain() prints
    joined.explain()
for line in plan.getvalue().splitlines():
    if "Broadcast" in line:
        print(line.strip()[:72])

**What to notice:** `F.column` behaves exactly like `F.col`, and `lit` repeats one constant on every row. A decimal
literal keeps `gross` exact; `F.lit(1.2)` would be a `double` and turn 84.50 into 101.39999999999999. The plan
contains a `BroadcastHashJoin` fed by a `BroadcastExchange`: Spark copied the small `customers` table to every
task instead of shuffling `orders` by `customer_id`.

## Choose the right data type

| type | holds | use for |
| --- | --- | --- |
| `int`, `bigint` | 32- and 64-bit integers | ids, counts |
| `double` | binary floating point | measurements, scores, ratios |
| `decimal(p, s)` | exact numbers with `p` digits, `s` after the point | **money**, quantities that must add up |
| `string` | text | names, codes |
| `boolean` | true / false | flags |
| `date` | a calendar day | business dates, partitions |
| `timestamp` | an instant, shown in the session time zone | event times |
| `timestamp_ntz` | a wall-clock time with no time zone | store opening hours, local schedules |
| `array`, `struct`, `map` | nested values | covered in [lesson 8](pyspark-08-nested-and-semi-structured-data.ipynb) |

A `double` stores numbers in binary, so most decimal fractions are approximations; sums of many of them drift.
A `decimal` stores them exactly, at the cost of a fixed number of digits.

**Where it matters**
- **Finance tables:** revenue reconciliations that must match to the cent.
- **Schemas for consumers:** downstream tools read your types; `bigint` vs `int` or `decimal` vs `double` is part of the contract.

**Watch out for**
- A decimal result can exceed its precision: Spark widens precision for arithmetic, up to 38 digits.
- Using `string` for everything postpones type errors to the consumer.

For example, `decimal(12,2)` allows ten digits before the decimal point and two after it. Choose precision and scale from the largest amount and smallest unit your source can represent. Keep identifiers like `"00042"` as strings when leading zeros matter. Use `printSchema()` after parsing or calculation to check the actual output type, especially when Spark has widened a decimal expression.

### Example

Add 0.1 and 0.2 as `double` and as `decimal`, and compare each result with 0.3.

In [ ]:
prices = spark.createDataFrame([(0.1, 0.2, Decimal("0.1"), Decimal("0.2"))], "a double, b double, c decimal(4,2), d decimal(4,2)")
result = prices.select(
    (F.col("a") + F.col("b")).alias("double_sum"),
    (F.col("a") + F.col("b") == 0.3).alias("double_equals_0_3"),
    (F.col("c") + F.col("d")).alias("decimal_sum"),
    (F.col("c") + F.col("d") == Decimal("0.3")).alias("decimal_equals_0_3"),
)
result.show()
result.printSchema()

**What to notice:** As doubles, 0.1 + 0.2 is `0.30000000000000004`, so it is not equal to 0.3: neither 0.1 nor 0.2 can be stored
exactly in binary. As decimals the sum is exactly `0.30`. The schema shows Spark widened the decimal result to
`decimal(5,2)` so the addition cannot overflow. Errors like this one, repeated over millions of rows, are why
revenue reconciliations fail on `double` columns.

## Cast types and handle bad values

Use `F.col("units_text").cast("int")` when invalid input should stop the pipeline. With ANSI mode enabled, as it is by default in Spark 4, malformed values and numeric overflow raise errors. The failure usually surfaces when an action evaluates the expression. Use `F.col("units_text").try_cast("int")` when malformed or out-of-range rows should be retained for review. Choose a supported conversion in both cases.

| Input | Function to reach for | Example |
| --- | --- | --- |
| Plain value with a known target type | `cast` or `try_cast` | `"4"` to an integer |
| Number with a formatting contract | `try_to_number` | `"1,204.50"` with `"9,999.99"` |
| Timestamp text with a known pattern | `try_to_timestamp` | `"31/01/2026 18:45"` with `"dd/MM/yyyy HH:mm"` |
| Ratio with a possible zero denominator | `try_divide` | A return rate for a group with zero orders |

Keep the raw and parsed columns long enough to tell an absent input from a failed conversion. If the raw value is present and parsing returns null, count or route the row as a reject. Decide how blanks should be treated before parsing. [Spark 4.0 PySpark try_cast reference](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.Column.try_cast.html)

Other tolerant arithmetic functions include `try_add`, `try_subtract`, and `try_multiply` for arithmetic overflow, and `try_sum` and `try_avg` for aggregate overflow. They follow the same pattern of returning null on the supported error. Track those nulls before applying a fallback so overflow stays visible.

### Example

Cast clean and dirty strings, see the error from a plain cast, and handle the same values with `try_` functions.

In [ ]:
raw = spark.createDataFrame(
    [("o1", "120.50", "4"), ("o2", "N/A", "0"), ("o3", "1,204.50", "2"), ("o4", None, "2")],
    "order_id string, amount_text string, units_text string",
)

spark.sparkContext.setLogLevel("OFF")  # the failure below is expected
try:
    raw.select(F.col("amount_text").cast("decimal(12,2)")).collect()
except Exception as exc:
    print("plain cast:", str(exc).splitlines()[0][:90])
finally:
    spark.sparkContext.setLogLevel("ERROR")

typed = raw.select(
    "order_id",
    "amount_text",
    F.col("amount_text").try_cast("decimal(12,2)").alias("try_cast"),
    F.try_to_number("amount_text", F.lit("9,999.99")).alias("try_to_number"),
    F.try_divide(F.lit(100), F.col("units_text").try_cast("int")).alias("per_unit"),
)
typed.withColumn(
    "plain_amount_rejected",
    F.col("amount_text").isNotNull() & F.col("try_cast").isNull(),
).orderBy("order_id").show()

**What to notice:** The plain cast failed on the whole job because of one value. `try_cast` returned null for both `N/A` and
`1,204.50`, since the comma is not part of a plain number. `try_to_number` with the format `9,999.99` parsed
`1,204.50` correctly and still returned null for `N/A`. `try_divide` returned null for the division by zero
instead of failing.

Rows `o2` and `o3` both fail the plain-decimal contract. The formatted parser accepts `o3`, so the reject definition depends on which source contract applies. Row `o4` is a missing input and the reject flag stays false.

### Handle arithmetic overflow

Add to the largest 32-bit integer, subtract from the smallest, and double the largest. Then try summing two large 64-bit values.

In [ ]:
limits = spark.createDataFrame(
    [(2147483647, -2147483648, 2)],
    "int_max int, int_min int, small int",
)
limits.select(
    F.try_add("int_max", F.lit(1)).alias("add_overflow"),
    F.try_subtract("int_min", F.lit(1)).alias("sub_overflow"),
    F.try_multiply("int_max", F.lit(2)).alias("mul_overflow"),
    F.try_multiply("small", F.lit(2)).alias("product"),
).show()

large_values = spark.createDataFrame(
    [(9223372036854775807,), (2,)], "value long",
)
large_values.agg(
    F.try_sum("value").alias("sum_long"),
    F.try_sum(F.col("value").cast("decimal(20,0)")).alias("sum_decimal"),
).show()
decimal_limits = spark.createDataFrame(
    [(D("9" * 38),), (D("9" * 38),)], "value decimal(38,0)",
)
decimal_limits.agg(F.try_avg("value").alias("avg_overflow")).show()

**Expected output**

```text
+------------+------------+------------+-------+
|add_overflow|sub_overflow|mul_overflow|product|
+------------+------------+------------+-------+
|        NULL|        NULL|        NULL|      4|
+------------+------------+------------+-------+

+--------+-------------------+
|sum_long|        sum_decimal|
+--------+-------------------+
|    NULL|9223372036854775809|
+--------+-------------------+

+------------+
|avg_overflow|
+------------+
|        NULL|
+------------+
```

**What to notice:** The 32-bit operations overflow and yield null. Widening the long values before summing preserves the total. The decimal average also returns null when its intermediate sum overflows.

## Handle nulls

Spark SQL uses three-valued logic. Ordinary comparisons such as `region == "North"` and `region != "North"` return null when the region is missing. A filter keeps only true rows. Negating a predicate with `~` leaves unknown values unknown. Combine Spark predicates with `&` and `|`, and parenthesize each comparison because Spark evaluates a column expression for every row.

| `region` | `region != "North"` | `~region.eqNullSafe("North")` |
| --- | --- | --- |
| North | false | false |
| South | true | true |
| null | null | true |

Use `isNull()` to find missing values. Use `eqNullSafe` when comparing two nullable values, such as before and after snapshots. Two nulls compare equal, and a null paired with a known value compares unequal. In a combined flag, `false & null` is false and `true & null` is null. [Spark SQL null semantics](https://spark.apache.org/docs/4.0.1/sql-ref-null-semantics.html)

| Need | Expression |
| --- | --- |
| Take the first available phone or default | `F.coalesce("mobile", "home", F.lit("unavailable"))` |
| Treat an empty value as missing | `F.nullif(F.trim("email"), F.lit(""))` |
| Fill selected fields according to a business rule | `df.fillna({"region": "unknown"})` |
| Drop rows with any missing required key | `df.dropna(subset=["customer_id", "order_id"])` |
| Drop rows where every candidate field is missing | `df.dropna(how="all", subset=["mobile", "home"])` |

`count("*")` counts rows. `count("region")` counts known regions. `sum` and `avg` skip nulls and return null when all inputs are null. For example, an order group with all amounts missing still has orders. Count rows to measure volume and count known amounts to measure completeness.

Null and floating-point `NaN` can represent different problems. Use `isnan` on a floating-point field to detect `NaN`, and `isNull` to detect SQL null. `nanvl(x, y)` replaces a `NaN` with `y`, while `coalesce(x, y)` supplies a fallback for null. Spark's `fillna` and `dropna` also handle `NaN` in floating-point columns.

### Example

Compare with and without null safety, fill and drop nulls, turn an empty string into null, and count with and without nulls.

In [ ]:
contacts = spark.createDataFrame(
    [(1, "North", "a@x.com"), (2, None, ""), (3, "South", None), (4, None, "d@x.com")],
    "id int, region string, email string",
)
contacts.select(
    "id", "region",
    (F.col("region") == F.lit(None)).alias("equals_null"),
    F.col("region").eqNullSafe(F.lit(None)).alias("null_safe"),
    F.coalesce("region", F.lit("unknown")).alias("region_or_default"),
    F.nullif("email", F.lit("")).alias("email_clean"),
).show()

print("dropna(subset=['region']):", contacts.dropna(subset=["region"]).count(), "rows")
contacts.fillna({"region": "unknown", "email": "none"}).show()
contacts.agg(F.count("*").alias("rows"), F.count("region").alias("with_region")).show()

region = F.col("region")
contacts.select(
    "id", "region",
    (region != "North").alias("ordinary_comparison"),
    (~region.eqNullSafe("North")).alias("include_unknowns"),
).orderBy("id").show()

**What to notice:** `region == null` is null for every row, even where region is null, so it can never select anything; only
`eqNullSafe` returns true for the missing regions. `nullif` turned customer 2's empty email into a real null.
`count("*")` counts 4 rows but `count("region")` only the 2 known regions.

A filter on `ordinary_comparison` keeps only the South row. A filter on `include_unknowns` keeps that row and both rows with a missing region. Choose the predicate based on whether the report should include customers whose region is unknown.

### Choose a fallback and drop only empty contact rows

Prefer mobile over home, treating empty or space-only strings as missing. Then remove rows with no remaining phone.

In [ ]:
phone_inputs = spark.createDataFrame(
    [(1, "555-0101", "555-0102"), (2, "  ", "555-0202"), (3, None, None)],
    "id int, mobile string, home string",
)
clean_phones = phone_inputs.select(
    "id",
    F.nullif(F.trim("mobile"), F.lit("")).alias("mobile"),
    F.nullif(F.trim("home"), F.lit("")).alias("home"),
)
clean_phones.select(
    "id", F.coalesce("mobile", "home", F.lit("unavailable")).alias("preferred"),
).orderBy("id").show()
clean_phones.dropna(how="all", subset=["mobile", "home"]).orderBy("id").show()

**Expected output**

```text
+---+-----------+
| id|  preferred|
+---+-----------+
|  1|   555-0101|
|  2|   555-0202|
|  3|unavailable|
+---+-----------+

+---+--------+--------+
| id|  mobile|    home|
+---+--------+--------+
|  1|555-0101|555-0102|
|  2|    NULL|555-0202|
+---+--------+--------+
```

**What to notice:** Row 2 survives and uses the home number. Row 3 receives a fallback in the first output and drops from the second. Apply the drop to the cleaned phone columns before replacing missing values with a label.

### Use the SQL-style null helpers

Code ported from SQL often uses these short forms. Each one can be written with `coalesce` or `when`, so pick the
one your team reads most easily:

| function | returns | same as |
| --- | --- | --- |
| `isnull(a)`, `isnotnull(a)` | true or false, never null | `a.isNull()`, `a.isNotNull()` |
| `nvl(a, b)`, `ifnull(a, b)` | `a`, or `b` when `a` is null | `coalesce(a, b)` |
| `nvl2(a, b, c)` | `b` when `a` is not null, otherwise `c` | `when(a.isNotNull(), b).otherwise(c)` |

In [ ]:
shipments = spark.createDataFrame(
    [(1, "DHL", "TRK-881"), (2, None, None), (3, "UPS", None)],
    "order_id int, carrier string, tracking string",
)
shipments.select(
    "order_id", "carrier", "tracking",
    F.isnull("tracking").alias("isnull"),
    F.isnotnull("tracking").alias("isnotnull"),
    F.nvl("carrier", F.lit("unassigned")).alias("nvl"),
    F.ifnull("carrier", F.lit("unassigned")).alias("ifnull"),
    F.nvl2("tracking", F.lit("in transit"), F.lit("awaiting label")).alias("nvl2"),
).orderBy("order_id").show()

**What to notice:** `isnull` and `isnotnull` always return true or false, so they are safe inside a filter.
`nvl` and `ifnull` give identical results, a fallback for the missing carrier. `nvl2` ignores the tracking value
itself and only asks whether it exists, so order 3 is still awaiting a label even though it has a carrier.

### Sample problem · Audit a noisy sensor feed

Given four sensor values `12.5`, null, `NaN`, and `0.0`, report the number of rows, missing readings, invalid readings, usable readings, and the average usable value. Zero is a valid measurement. Predict why `count("reading")` alone gives the wrong usability count.

In [ ]:
sensor_readings = spark.createDataFrame(
    [(12.5,), (None,), (float("nan"),), (0.0,)],
    "reading double",
)
usable_reading = F.nanvl("reading", F.lit(None).cast("double"))
sensor_readings.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("reading").isNull()).alias("missing"),
    F.count_if(F.isnan("reading")).alias("invalid"),
    F.count(usable_reading).alias("usable"),
    F.avg(usable_reading).alias("avg_usable"),
).show()

**Expected result:** 4 rows, 1 missing reading, 1 invalid reading, 2 usable readings, and an average of 6.25. The original column has three non-null values because `NaN` counts as present. The expression passed to `avg` converts it to null so Spark excludes it.

## Write conditional logic

Use `F.when(condition, value).when(...).otherwise(default)` for a per-row classification. Spark picks the first true condition, so put a narrow rule before a broader rule that would also match. A null condition does not match, and omitting `otherwise` gives unmatched rows null. If a missing amount should be labeled `"unknown"`, handle it in an explicit first branch. Keep result branches in compatible types.

Use `isin("paid", "shipped")` to test membership in a small fixed list. For rules maintained in a separate table, a lookup join lets the data carry the mapping. Use `greatest` or `least` when choosing among several columns in the same row. Both skip nulls and return null if every candidate is null.

A conditional expression can feed an aggregate. Use `count_if` to count true predicates, or use `sum(when(...).otherwise(...))` when each qualifying row contributes an amount. Decide whether other rows should contribute zero or represent missing data.

### Example

Assign size tiers with `when`, and show what happens to rows that match no branch when `otherwise` is missing.

In [ ]:
orders.select(
    "order_id", "status", "amount",
    F.when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
    F.when(F.col("status").isin("paid", "shipped"), True).alias("revenue_flag"),
).orderBy("order_id").show()

**What to notice:** Only order 9 is `enterprise`, because the `>= 1000` branch is tested first. `revenue_flag` has no `otherwise`, so
placed and returned orders get null instead of false; a later `where(~revenue_flag)` would miss them.

In [ ]:
is_paid = F.col("status") == "paid"
orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"),
    F.count_if(is_paid).alias("paid_orders"),
    F.sum(F.when(is_paid, F.col("amount")).otherwise(D("0.00"))).alias("paid_amount"),
).orderBy("customer_id").show()

Customer 4 has two orders and zero paid orders. The explicit zero branch gives that group a paid amount of `0.00`. Leaving off `otherwise` would give the group an all-null input to `sum` and a null paid amount.

## Sort with nulls first or last

`F.asc("col")` and `F.desc("col")` (or the `Column` methods `.asc()` and `.desc()`) set the direction of an
`orderBy`. Spark puts nulls **first** in ascending order and **last** in descending order. To choose explicitly,
use `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first` or `desc_nulls_last`.

**Where it matters**
- **Reports:** rows with an unknown region should not crowd the top of a list.
- **Window ordering:** `row_number` over a column with nulls ranks them first unless you say otherwise.

**Watch out for**
- Engines differ: PostgreSQL puts nulls last in ascending order, so a query ported between engines can change order.
- Add a unique tie-breaker column when the output order must be the same on every run.

### Example

Sort customers by region in each of the six ways and see where customer 4's missing region lands.

In [ ]:
def regions(sort_col):
    return [row.region for row in customers.orderBy(sort_col, "customer_id").collect()]

for label, sort_col in [
    ("asc", F.asc("region")),
    ("asc_nulls_first", F.asc_nulls_first("region")),
    ("asc_nulls_last", F.asc_nulls_last("region")),
    ("desc", F.desc("region")),
    ("desc_nulls_first", F.desc_nulls_first("region")),
    ("desc_nulls_last", F.desc_nulls_last("region")),
]:
    print(f"{label:17}", regions(sort_col))

**What to notice:** Plain `asc` puts `None` first and plain `desc` puts it last. `asc_nulls_first` and
`desc_nulls_last` restate those defaults, while `asc_nulls_last` and `desc_nulls_first` move the null to the other
end. `customer_id` breaks ties between equal regions so every run prints the same order.

## Work with strings

Start with the simplest operation that matches the source format. For a case-insensitive email key, `lower(trim(email))` handles case and outer spaces. Preserve the raw value for troubleshooting, and apply the same normalization on both join inputs. Use `nullif` if a blank result should count as missing.

| Task | Functions | Behavior to check |
| --- | --- | --- |
| Clean case and outer spaces | `trim`, `lower`, `upper`, `initcap` | Inner spaces remain after `trim` |
| Combine fields | `concat`, `concat_ws` | `concat` propagates a null input. `concat_ws` skips null fields but keeps empty strings |
| Read fixed positions or format an ID | `substring`, `length`, `lpad` | Substring positions start at 1. `lpad` truncates strings longer than the requested width |
| Split a delimited field | `split` | The separator is a regex, so a literal pipe needs `r"\|"` |
| Extract or replace a pattern | `regexp_extract`, `regexp_replace` | Group 0 is the full match. An unmatched extraction on a non-null input gives an empty string |

For a code such as `UK-LON-0042`, anchoring the regex with `^` and `$` validates the entire code. Wrap the extraction in `nullif(..., F.lit(""))` if a nonmatch should be missing. If an ID is part of a key, check its length before padding so a long ID cannot be silently shortened.

| When you need to… | Reach for |
| --- | --- |
| Read the last token after a delimiter | `substring_index("code", "-", -1)`. Positive counts read from the left |
| Check for a substring or prefix | `col.contains("red")`, `col.startswith("SKU-")`, or `col.endswith(".csv")` with literal text |
| Match a SQL wildcard or regex | `col.like("SKU-%")` uses SQL wildcards. `col.rlike(r"^SKU-[0-9]+$")` uses a regex |
| Replace fixed characters | `translate("phone", "()- ", "")` removes those individual characters. Use `regexp_replace` for patterns |
| Find every occurrence or count matches | `regexp_extract_all("text", F.lit(r"[0-9]+"), 0)` returns matches as an array. `regexp_count` returns their count |
| Produce a formatted label | `format_string("SKU-%05d", "id")` formats an integer with at least five digits |

### Example

Normalize emails, build a display name, pad an id, extract parts with regular expressions, and split a code.

In [ ]:
people = spark.createDataFrame(
    [("  Ada@Example.COM ", "ada", "lovelace", 7, "UK-LON-0042"),
     ("grace@example.org", "grace", None, 42, "US-NYC-0007"),
     ("lin@example.org", "lin", "chen", 108, "unassigned")],
    "email string, first string, last string, id int, location string",
)
people.select(
    F.lower(F.trim("email")).alias("email"),
    F.concat(F.initcap("first"), F.lit(" "), F.initcap("last")).alias("concat"),
    F.concat_ws(" ", F.initcap("first"), F.initcap("last")).alias("concat_ws"),
    F.lpad(F.col("id").cast("string"), 5, "0").alias("padded_id"),
    F.nullif(
        F.regexp_extract("location", r"^([A-Z]{2})-([A-Z]{3})-(\d+)$", 2), F.lit("")
    ).alias("city"),
    F.regexp_replace("location", r"\d", "#").alias("masked"),
    F.split("location", "-").alias("parts"),
).show(truncate=False)

**What to notice:** Grace has no last name: `concat` returned null for her, while `concat_ws` skipped the null and returned
`Grace`. `regexp_extract` with group 2 pulled out the city code, and `split` returned an array you can index
later. `lpad` made every id five characters wide.

The `unassigned` value has no city match. `nullif` turns the empty extraction into null, which can be counted or routed as a missing code.

### Check string prefixes, suffixes, and patterns

Compare a valid SKU with a filename, an invalid SKU with the right prefix, and a missing input. Apply both SQL wildcard and regular expression tests.

In [ ]:
labels = spark.createDataFrame(
    [(1, "SKU-042"), (2, "red.csv"), (3, "SKU-ABC"), (4, None)], "id int, label string",
)
label = F.col("label")
labels.select(
    "id", "label",
    label.contains("red").alias("has_red"),
    label.startswith("SKU-").alias("sku_prefix"),
    label.endswith(".csv").alias("csv_suffix"),
    label.like("SKU-%").alias("like_sku"),
    label.rlike(r"^SKU-[0-9]+$").alias("regex_sku"),
).orderBy("id").show()

**Expected output**

```text
+---+-------+-------+----------+----------+--------+---------+
| id|  label|has_red|sku_prefix|csv_suffix|like_sku|regex_sku|
+---+-------+-------+----------+----------+--------+---------+
|  1|SKU-042|  false|      true|     false|    true|     true|
|  2|red.csv|   true|     false|      true|   false|    false|
|  3|SKU-ABC|  false|      true|     false|    true|    false|
|  4|   NULL|   NULL|      NULL|      NULL|    NULL|     NULL|
+---+-------+-------+----------+----------+--------+---------+
```

**What to notice:** Every predicate returns null for the missing input. The wildcard matches any suffix after the prefix, while the regex requires digits.

### Extract tokens, count matches, and format IDs

Use a label with two digit groups and one with no digits. Apply literal character removal to a phone number.

In [ ]:
string_inputs = spark.createDataFrame(
    [(7, "A12-B34", "(512) 555-0100"), (42, "NONE", None)],
    "id int, label string, phone string",
)
digits_pattern = F.lit(r"[0-9]+")
string_inputs.select(
    "id",
    F.substring("label", 1, 3).alias("first_three"),
    F.length("label").alias("length"),
    F.substring_index("label", "-", -1).alias("last_token"),
    F.translate("phone", "()- ", "").alias("phone_digits"),
    F.regexp_extract_all("label", digits_pattern, 0).alias("matches"),
    F.regexp_count("label", digits_pattern).alias("match_count"),
    F.format_string("SKU-%05d", "id").alias("formatted_id"),
).orderBy("id").show(truncate=False)

**Expected output**

```text
+---+-----------+------+----------+------------+--------+-----------+------------+
|id |first_three|length|last_token|phone_digits|matches |match_count|formatted_id|
+---+-----------+------+----------+------------+--------+-----------+------------+
|7  |A12        |7     |B34       |5125550100  |[12, 34]|2          |SKU-00007   |
|42 |NON        |4     |NONE      |NULL        |[]      |0          |SKU-00042   |
+---+-----------+------+----------+------------+--------+-----------+------------+
```

**What to notice:** The first position is 1. When the delimiter is absent, the last token is the original string. A nonmatching, non-null label gives an empty match array and a count of zero.

### Trim one side, slice, pad and format numbers

| function | does |
| --- | --- |
| `ltrim`, `rtrim` | remove spaces on the left or right only |
| `left(s, n)`, `right(s, n)` | the first or last `n` characters |
| `split_part(s, delim, n)` | the `n`th piece after splitting on a literal delimiter (no regex, no array) |
| `rpad(s, len, pad)` | pad on the right to a fixed width; like `lpad`, it truncates longer values |
| `replace(s, search, repl)` | replace every occurrence of a substring, without regex |
| `format_number(x, d)` | text such as `1,234,567.89` with `d` decimals |

In [ ]:
codes = spark.createDataFrame(
    [("  sku-4001  ", "Kitchen|Knives|Chef", 1234567.891), ("SKU-4002", "Garden|Hoses", 89.5)],
    "raw_sku string, path string, price double",
)
codes.select(
    F.upper(F.trim("raw_sku")).alias("sku"),
    F.length("raw_sku").alias("raw_len"),
    F.length(F.ltrim("raw_sku")).alias("ltrim_len"),
    F.length(F.rtrim("raw_sku")).alias("rtrim_len"),
    F.left(F.trim("raw_sku"), F.lit(3)).alias("left_3"),
    F.right(F.trim("raw_sku"), F.lit(4)).alias("right_4"),
    F.rpad(F.trim("raw_sku"), 10, ".").alias("rpad_10"),
).show(truncate=False)

codes.select(
    F.split_part("path", F.lit("|"), F.lit(1)).alias("top_level"),
    F.split_part("path", F.lit("|"), F.lit(-1)).alias("leaf"),
    F.replace("path", F.lit("|"), F.lit(" > ")).alias("breadcrumb"),
    F.format_number("price", 2).alias("display_price"),
).show(truncate=False)

**What to notice:** The first SKU is 12 characters raw; `ltrim` and `rtrim` each remove the two spaces on one side,
so both leave 10. `split_part` takes a piece by position without the regex escaping `split` needs for `|`, and a
negative position counts from the end. `replace` swapped one character for a three-character string, which
`translate` cannot do. `format_number` returns text, so format only for display, at the very end.

### Sample problem · Normalize vendor SKU codes

A vendor sends `" eu / 00042 "`, `"us/00007"`, `"APAC-123"`, and null. Accept a two-letter market, a slash, and exactly five digits, allowing outer spaces and spaces around the slash. Return the raw value, a canonical key such as `EU-00042`, and a flag for supplied values that failed validation. Keep leading zeros.

In [ ]:
vendor_skus = spark.createDataFrame(
    [(1, " eu / 00042 "), (2, "us/00007"), (3, "APAC-123"), (4, None)],
    "row_id int, raw_sku string",
)
normalized_sku = F.regexp_replace(F.upper(F.trim("raw_sku")), r"\s*/\s*", "/")
sku_pattern = r"^([A-Z]{2})/([0-9]{5})$"
canonical_sku = F.when(
    normalized_sku.rlike(sku_pattern),
    F.concat_ws("-", F.substring_index(normalized_sku, "/", 1), F.substring_index(normalized_sku, "/", -1)),
)
vendor_skus.withColumn("sku", canonical_sku).select(
    "row_id", "raw_sku", "sku",
    (F.col("raw_sku").isNotNull() & F.col("sku").isNull()).alias("rejected"),
).orderBy("row_id").show(truncate=False)

**Expected result:** `EU-00042` and `US-00007` are accepted. `APAC-123` is rejected and the missing input is counted separately. The validation guard prevents the string functions from creating a convincing key from malformed input.

## Round and compute numbers

`round(col, n)` rounds half up (2.5 becomes 3); `bround` rounds half to even (2.5 becomes 2), which can reduce
bias in large sums. `ceil`, `floor`, `abs` and `pow` do what their names say. Keep money in `decimal` and
round only for display or at a defined business step, not at every intermediate calculation.

**Where it matters**
- **Financial reporting rules,** which often specify the rounding mode.
- **Percentages and ratios** computed from integer counts.

**Watch out for**
- Integer division does not truncate in Spark: `7 / 2` is `3.5`. Use `F.floor` to round down or SQL `div` to truncate the fractional part.
- Rounding before summing gives a different total than summing before rounding.

| Need | Functions and tradeoffs |
| --- | --- |
| Work with magnitudes and bounds | `abs` measures magnitude. `greatest` and `least` can apply lower and upper bounds, but skip null arguments, so guard missing inputs when the result should stay null |
| Handle signed remainders | `pmod(x, 7)` gives a nonnegative remainder for a positive divisor, useful for weekday rotation and bucket IDs |
| Compare to a threshold relative to the data | `percentile_approx("amount", 0.95)` estimates the 95th percentile. `median("amount")` gives the median |
| Report variation | `stddev_samp` estimates sample standard deviation. `stddev_pop` describes the whole population |
| Count unique values | `count_distinct` gives the exact count. `approx_count_distinct` estimates it with bounded relative standard deviation controlled by `rsd` |

### Example

Compare `round` and `bround` on halves, and divide integers.

In [ ]:
halves = spark.createDataFrame([(Decimal("0.5"),), (Decimal("1.5"),), (Decimal("2.5"),)], "value decimal(3,1)")
halves.select(
    "value", F.round("value").alias("round"), F.bround("value").alias("bround"),
    F.ceil("value").alias("ceil"), F.floor("value").alias("floor"),
).show()
spark.range(1).select((F.lit(7) / 2).alias("7 / 2"), F.expr("7 div 2").alias("7 div 2")).show()

**What to notice:** On the positive halves shown, `round` gives 1, 2, 3 and `bround` gives 0, 2, 2. Half-up rounds exact ties away from zero. Half-even goes to the nearest even value and can reduce systematic bias when many values land on ties. Use the mode specified by the business rule. Ordinary division of the two integers gives 3.5, while `div` gives 3. For negative values, `floor` rounds toward negative infinity and `div` truncates toward zero.

### Bound a number and assign a remainder bucket

Clamp each known input into the range 0 to 10, while preserving a missing input.

In [ ]:
number_inputs = spark.createDataFrame(
    [(1, -3), (2, 5), (3, 12), (4, None)], "id int, x int",
)
x = F.col("x")
number_inputs.select(
    "id", "x",
    F.abs(x).alias("abs"),
    F.pow(x, 2).alias("squared"),
    F.pmod(x, F.lit(7)).alias("mod7"),
    F.greatest(x, F.lit(0)).alias("at_least_zero"),
    F.when(x.isNotNull(), F.least(F.greatest(x, F.lit(0)), F.lit(10))).alias("bounded"),
).orderBy("id").show()

**Expected output**

```text
+---+----+----+-------+----+-------------+-------+
| id|   x| abs|squared|mod7|at_least_zero|bounded|
+---+----+----+-------+----+-------------+-------+
|  1|  -3|   3|    9.0|   4|            0|      0|
|  2|   5|   5|   25.0|   5|            5|      5|
|  3|  12|  12|  144.0|   5|           12|     10|
|  4|NULL|NULL|   NULL|NULL|            0|   NULL|
+---+----+----+-------+----+-------------+-------+
```

**What to notice:** The remainder of −3 under a positive divisor of 7 is 4 with `pmod`. The unguarded lower bound gives 0 for null because `greatest` skips nulls, while the guarded clamp keeps null.

### Summarize a small distribution

Compare the exact and estimated distinct counts and the sample and population measures on values 10, 20, 20, and 30.

In [ ]:
distribution = spark.createDataFrame([(10,), (20,), (20,), (30,)], "value int")
distribution.agg(
    F.percentile_approx("value", 0.95).alias("p95_approx"),
    F.median("value").alias("median"),
    F.round(F.stddev_samp("value"), 2).alias("sample_sd"),
    F.round(F.stddev_pop("value"), 2).alias("pop_sd"),
    F.count_distinct("value").alias("distinct_exact"),
    F.approx_count_distinct("value").alias("distinct_approx"),
).show()

**Expected output**

```text
+----------+------+---------+------+--------------+---------------+
|p95_approx|median|sample_sd|pop_sd|distinct_exact|distinct_approx|
+----------+------+---------+------+--------------+---------------+
|        30|  20.0|     8.16|  7.07|             3|              3|
+----------+------+---------+------+--------------+---------------+
```

**What to notice:** There are three distinct values. The estimate agrees on this input. The sample standard deviation is larger because it uses a denominator of 3 instead of 4.

### Roots, logarithms and random numbers

`sqrt`, `exp` and `log` behave as in any math library. `log(x)` is the natural logarithm; `log(base, x)` takes the
base first. A log of zero or a negative number returns null rather than failing. `rand(seed)` returns a uniform
random number in [0, 1), useful for sampling or splitting rows into test and control groups.

In [ ]:
growth = spark.createDataFrame([(1, 100.0), (2, 0.0), (3, 2.25)], "id int, value double")
growth.select(
    "id", "value",
    F.sqrt("value").alias("sqrt"),
    F.round(F.log("value"), 3).alias("ln"),
    F.log(10.0, "value").alias("log10"),
    F.round(F.exp(F.log("value")), 2).alias("exp_of_ln"),
).orderBy("id").show()

spark.range(4, numPartitions=1).select(
    "id", F.round(F.rand(seed=7), 3).alias("rand"),
    F.when(F.rand(seed=7) < 0.5, "control").otherwise("test").alias("group"),
).show()

**What to notice:** The log of 100 is about 4.605 natural and exactly 2.0 in base 10, and `exp` undoes `log`. The
log of 0 is null, not an error or minus infinity, so count those nulls if zeros are possible. With the same seed
and partitioning, `rand` repeats the same numbers on every run, which keeps a test/control split reproducible.

### Sample problem · Compare return rates by customer

From `orders`, report total orders, returned orders, percentage returned, and the largest order amount for each customer with orders. Round the percentage to one decimal place. Customer 1 has one return in three orders. Customer 4 has one in two.

In [ ]:
return_summary = orders.groupBy("customer_id").agg(
    F.count("*").alias("order_count"),
    F.count_if(F.col("status") == "returned").alias("return_count"),
    F.max("amount").alias("largest_order"),
)
return_summary.withColumn(
    "return_pct", F.round(F.try_divide("return_count", "order_count") * 100, 1),
).orderBy(F.desc("return_pct"), "customer_id").show()

**Expected result:** Customers 4 and 1 lead at 50.0% and 33.3% respectively. Their largest orders are 1180.00 and 120.00. Customer 2 has orders and a zero return rate. A customer with zero total orders would have an undefined rate, which `try_divide` represents as null.

## Work with dates and timestamps

Parse source text into a typed value before filtering or grouping. Use `date` for a calendar day and `timestamp` for an event instant. Use `timestamp_ntz` when the source represents a wall-clock reading without enough information to place it on a global timeline.

| Need | Functions | Detail |
| --- | --- | --- |
| Parse a date or timestamp | `to_date`, `to_timestamp`, `try_to_timestamp` | Use the source pattern and a tolerant parser when bad input should become null |
| Report elapsed calendar days | `datediff(end, start)` | Argument order determines the sign |
| Move by days or months | `date_add`, `add_months`, `last_day` | Adding a month to January 31 lands on February's last valid day |
| Group by calendar period | `trunc`, `date_trunc` | `trunc(date, "month")` returns a date. `date_trunc("month", ts)` returns a timestamp |
| Display a date or derive a label | `date_format`, `year`, `month`, `dayofweek` | `date_format` returns a string. `dayofweek` gives Sunday 1 and Saturday 7 |
| Read Unix epoch milliseconds | `timestamp_millis` | Check the input unit, since epoch seconds need `timestamp_seconds` |

Pattern letters are case-sensitive. In `"yyyy-MM-dd HH:mm"`, `MM` is month and `mm` is minute. If two formats are accepted, try each explicitly in a documented order and keep invalid rows visible. For bounded reporting periods, apply both boundaries, such as July 1 inclusive and January 1 exclusive for the second half of the year.

The session time zone controls how Spark displays timestamps and derives dates from them. This notebook pins the session to UTC. For a Chicago business date, it shifts the instant to Chicago wall-clock time with `from_utc_timestamp`, then takes the date in the UTC-pinned session. Keep the original timestamp for event comparisons and use the derived date for reporting. A named zone such as `America/Chicago` accounts for daylight saving time. For source strings with an offset, parse the offset into the instant before deriving the business date. [Spark 4.0 PySpark from_utc_timestamp reference](https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_utc_timestamp.html)

| Calendar task | Comparison |
| --- | --- |
| Build a date from separate integer fields | `make_date(year, month, day)` builds a date. `try_make_timestamp` is an option for assembling an instant from fields where invalid components should return null |
| Label a week or quarter | `weekofyear` uses ISO week numbering. Near New Year, the calendar year and ISO week-year can differ. A week-start date is a clear grouping key. `quarter` returns 1 through 4 |
| Measure date difference or timestamp duration | `datediff` counts date boundaries. `timestamp_diff("HOUR", start, end)` returns a whole-unit difference. Use an adequate unit when fractional durations matter |
| Convert an epoch field | Pair `timestamp_seconds`, `timestamp_millis`, or `timestamp_micros` with the source unit. `unix_seconds`, `unix_millis`, and `unix_micros` convert an instant back to epoch counts |
| Convert a local wall-clock timestamp | `convert_timezone(source_zone, target_zone, local_timestamp)` expects a timestamp without time zone and returns a wall-clock value in the target zone |

See the [Spark datetime pattern reference](https://spark.apache.org/docs/4.0.1/sql-ref-datetime-pattern.html) when a source specifies its own date or timestamp format.

### Example

Parse European-style dates, derive calendar fields, and convert order 12 from UTC to Chicago time.

In [ ]:
deliveries = spark.createDataFrame([("03/01/2026", "2026-01-31 18:45")], "day_text string, ts_text string")
deliveries.select(
    F.to_date("day_text", "dd/MM/yyyy").alias("day"),
    F.to_timestamp("ts_text", "yyyy-MM-dd HH:mm").alias("ts"),
    F.date_format(F.to_date("day_text", "dd/MM/yyyy"), "EEEE d MMMM yyyy").alias("pretty"),
    F.last_day(F.to_date("day_text", "dd/MM/yyyy")).alias("month_end"),
    F.add_months(F.to_date("day_text", "dd/MM/yyyy"), 1).alias("next_month"),
).show(truncate=False)

orders.where("order_id in (11, 12)").select(
    "order_id", "order_ts",
    F.to_date("order_ts").alias("utc_date"),
    F.from_utc_timestamp("order_ts", "America/Chicago").alias("chicago_time"),
    F.to_date(F.from_utc_timestamp("order_ts", "America/Chicago")).alias("chicago_date"),
    F.dayofweek("order_ts").alias("day_of_week"),
    F.datediff(F.lit("2026-01-31").cast("date"), F.to_date("order_ts")).alias("days_to_month_end"),
).show()

**What to notice:** `03/01/2026` became 3 January because the format says day first. Order 12 was placed at 00:30 UTC on 14 January,
which is still 13 January in Chicago, so the two business dates differ. `dayofweek` counts from Sunday = 1.

In [ ]:
date_inputs = spark.createDataFrame(
    [("2026-01-31",), ("01/31/2026",), ("2026-02-30",), (None,)],
    "date_text string",
)
parsed_date = F.to_date(F.coalesce(
    F.try_to_timestamp("date_text", F.lit("yyyy-MM-dd")),
    F.try_to_timestamp("date_text", F.lit("MM/dd/yyyy")),
))
date_inputs.select("date_text", parsed_date.alias("business_date")).show()

Both valid inputs yield January 31. February 30 and the missing input yield null. As with numeric casts, keep the source column until you have counted invalid supplied values separately from missing inputs.

### Build dates and inspect calendar boundaries

Use January 31 and an early Saturday in January to see end-of-month arithmetic and the ISO week boundary.

In [ ]:
date_parts = spark.createDataFrame(
    [(1, 2026, 1, 31), (2, 2027, 1, 2)], "id int, y int, m int, d int",
)
calendar_days = date_parts.withColumn("day", F.make_date("y", "m", "d"))
calendar_days.select(
    "id", "day",
    F.date_add("day", 1).alias("next_day"),
    F.datediff("day", F.trunc("day", "month")).alias("days_from_month_start"),
    F.trunc("day", "month").alias("month_start"),
    F.year("day").alias("year"), F.month("day").alias("month"),
    F.dayofweek("day").alias("weekday"), F.weekofyear("day").alias("iso_week"),
    F.quarter("day").alias("quarter"),
).orderBy("id").show()

**Expected output**

```text
+---+----------+----------+---------------------+-----------+----+-----+-------+--------+-------+
| id|       day|  next_day|days_from_month_start|month_start|year|month|weekday|iso_week|quarter|
+---+----------+----------+---------------------+-----------+----+-----+-------+--------+-------+
|  1|2026-01-31|2026-02-01|                   30| 2026-01-01|2026|    1|      7|       5|      1|
|  2|2027-01-02|2027-01-03|                    1| 2027-01-01|2027|    1|      7|      53|      1|
+---+----------+----------+---------------------+-----------+----+-----+-------+--------+-------+
```

**What to notice:** January 31 plus one day is February 1. January 2, 2027 is a Saturday in ISO week 53 of the preceding ISO week-year.

### Build and convert timestamps

Assemble two UTC instants from numeric components, one valid and one invalid. Convert a separate Chicago wall-clock reading to UTC and compare a timestamp duration to a date boundary.

In [ ]:
timestamp_parts = spark.createDataFrame(
    [(1, 28), (2, 30)], "id int, feb_day int",
)
timestamp_parts.select(
    "id",
    F.try_make_timestamp(
        F.lit(2026), F.lit(2), "feb_day", F.lit(18), F.lit(0), F.lit(0), F.lit("UTC")
    ).alias("built_utc"),
).orderBy("id").show()

wall_clock = F.expr("TIMESTAMP_NTZ '2026-01-13 23:30:00'")
start = F.expr("TIMESTAMP '2026-01-13 23:30:00'")
end = F.expr("TIMESTAMP '2026-01-14 01:00:00'")
spark.range(1).select(
    F.convert_timezone(F.lit("America/Chicago"), F.lit("UTC"), wall_clock).alias("as_utc_clock"),
    F.timestamp_diff("HOUR", start, end).alias("whole_hours"),
    F.timestamp_diff("MINUTE", start, end).alias("minutes"),
    F.datediff(F.to_date(end), F.to_date(start)).alias("date_boundaries"),
).show()

**Expected output**

```text
+---+-------------------+
| id|          built_utc|
+---+-------------------+
|  1|2026-02-28 18:00:00|
|  2|               NULL|
+---+-------------------+

+-------------------+-----------+-------+---------------+
|       as_utc_clock|whole_hours|minutes|date_boundaries|
+-------------------+-----------+-------+---------------+
|2026-01-14 05:30:00|          1|     90|              1|
+-------------------+-----------+-------+---------------+
```

**What to notice:** February 30 yields null. Chicago 23:30 is 05:30 UTC the next day. A 90-minute duration gives one whole hour and crosses one date boundary.

### Convert between timestamps and epoch units

Start from the same instant expressed as seconds, milliseconds, and microseconds after the Unix epoch.

In [ ]:
spark.range(1).select(
    F.timestamp_seconds(F.lit(D("1.234567"))).alias("from_seconds"),
    F.timestamp_millis(F.lit(1234)).alias("from_millis"),
    F.timestamp_micros(F.lit(1234567)).alias("from_micros"),
).show(truncate=False)

instant = F.timestamp_micros(F.lit(1234567))
spark.range(1).select(
    F.unix_seconds(instant).alias("seconds"),
    F.unix_millis(instant).alias("millis"),
    F.unix_micros(instant).alias("micros"),
).show()

**Expected output**

```text
+--------------------------+-----------------------+--------------------------+
|from_seconds              |from_millis            |from_micros               |
+--------------------------+-----------------------+--------------------------+
|1970-01-01 00:00:01.234567|1970-01-01 00:00:01.234|1970-01-01 00:00:01.234567|
+--------------------------+-----------------------+--------------------------+

+-------+------+-------+
|seconds|millis| micros|
+-------+------+-------+
|      1|  1234|1234567|
+-------+------+-------+
```

**What to notice:** Milliseconds cannot carry the remaining 567 microseconds in the first output. Converting the full timestamp to integral seconds or milliseconds discards finer precision.

### Read time parts, step through the calendar, and convert to UTC

- **Parts:** `dayofmonth`, `hour` and `minute` complete `year`, `month` and `quarter`.
- **Steps:** `date_sub(d, n)` goes back `n` days; `next_day(d, "Mon")` finds the next Monday, always after `d`;
  `months_between(a, b)` returns fractional months between two dates.
- **Epochs:** `unix_timestamp(ts)` gives seconds since 1970-01-01 UTC; `from_unixtime(seconds, fmt)` formats them as
  text in the session time zone.
- **Time zones:** `to_utc_timestamp(ts, zone)` reads a wall-clock time in `zone` and returns the UTC instant, the
  reverse of `from_utc_timestamp`.

In [ ]:
picked = orders.where("order_id IN (1, 7, 12)")
picked.select(
    "order_id", "order_ts",
    F.dayofmonth("order_ts").alias("day"),
    F.hour("order_ts").alias("hour"),
    F.minute("order_ts").alias("minute"),
    F.date_sub(F.to_date("order_ts"), 7).alias("week_before"),
    F.next_day(F.to_date("order_ts"), "Mon").alias("next_monday"),
    F.round(F.months_between(F.lit("2026-03-31").cast("date"), F.to_date("order_ts")), 2).alias("months_to_q_end"),
).orderBy("order_id").show()

picked.select(
    "order_id",
    F.unix_timestamp("order_ts").alias("epoch_s"),
    F.from_unixtime(F.unix_timestamp("order_ts"), "yyyy-MM-dd HH:mm").alias("from_epoch"),
).orderBy("order_id").show()

store_times = spark.createDataFrame([("Chicago", "2026-01-13 18:30:00")], "store string, local_text string")
store_times.select(
    "store", "local_text",
    F.to_utc_timestamp(F.to_timestamp("local_text"), "America/Chicago").alias("utc_instant"),
).show()

**What to notice:** 5 January 2026 is a Monday, yet `next_day` returns the 12th for order 1: it always moves
strictly forward. `months_between` counts fractional months, so mid-month dates give non-whole values.
`from_unixtime` reproduces the original time because the session time zone is UTC. A wall-clock time of 18:30 in
Chicago (UTC-6 in January) is 00:30 UTC the next day: the same instant as order 12.

### Bucket events by time, and stamp the current date

`F.window(ts, "1 day")` assigns each row to a fixed time bucket and returns a struct with `start` and `end`.
Group by it for counts per hour or per day; add a slide (`F.window(ts, "1 day", "12 hours")`) for overlapping
buckets. This grouping `window` is unrelated to the `Window` used by window functions.

`current_date()` and `current_timestamp()` return the date and instant the query started, the same value on
every row of one query.

In [ ]:
daily = orders.groupBy(F.window("order_ts", "1 day")).agg(
    F.count("*").alias("orders"), F.sum("amount").alias("revenue")
)
daily.select("window.start", "window.end", "orders", "revenue").orderBy("start").show(4)

stamped = spark.range(3).select("id", F.current_date().alias("today"), F.current_timestamp().alias("now"))
stamped.printSchema()
print("distinct values of now:", stamped.select("now").distinct().count())

**What to notice:** Each bucket runs from midnight to midnight UTC, and only days with orders appear: a day with
no orders is missing, not zero. `today` is a `date` and `now` a `timestamp`, and all three rows share one `now`.
Their values change on every run, which is why a rerunnable job should take its business date as a parameter.

### Sample problem · Report Chicago business weeks

Use orders 11 and 12. Derive their Chicago calendar dates from the UTC order timestamps and find the start of each Chicago business week. By the UTC calendar the orders fall on different days. Are they on different days in Chicago? What would break if you applied `to_date` before the time-zone adjustment?

In [ ]:
chicago_order_time = F.from_utc_timestamp("order_ts", "America/Chicago")
orders.where(F.col("order_id").isin(11, 12)).select(
    "order_id",
    F.to_date("order_ts").alias("utc_day"),
    F.to_date(chicago_order_time).alias("chicago_day"),
    F.to_date(F.date_trunc("week", chicago_order_time)).alias("chicago_week_start"),
).orderBy("order_id").show()

**Expected result:** The UTC dates are January 13 and 14. Both Chicago dates are January 13 and the Chicago week starts on Monday, January 12. Converting to a date first would discard the hour needed to decide which local day an order belongs to.

## Aggregate functions

An aggregate function reduces many rows to one value: across the whole DataFrame with `df.agg(...)`, or per group
with `df.groupBy(...).agg(...)`. [Lesson 7](pyspark-07-aggregations-and-windows.ipynb) covers grouping, pivots and
subtotals; this section is a tour of the functions themselves.

| function | returns |
| --- | --- |
| `count("*")`, `count(col)` | rows; non-null values |
| `count_distinct(col)`, `approx_count_distinct(col)` | exact distinct count; a fast estimate |
| `sum`, `avg` / `mean`, `min`, `max` | the usual summaries (`mean` is an alias of `avg`) |
| `sum_distinct(col)` | the sum of the distinct values |
| `count_if(cond)` | rows where the condition is true |
| `median`, `percentile_approx(col, p)` | exact median; approximate percentile |
| `stddev`, `variance`, `corr(a, b)` | sample standard deviation and variance; Pearson correlation |
| `collect_list`, `collect_set` | all values, or the distinct values, as an array |
| `first`, `last` | the first or last value Spark happens to see |
| `max_by(x, y)`, `min_by(x, y)` | the value of `x` on the row where `y` is largest or smallest |

**Where it matters**
- **Profiling:** row counts, distinct keys and ranges before you design a table.
- **Customer metrics:** biggest order, first order, statuses seen, number of paid orders.

**Watch out for**
- Every aggregate except `count("*")` skips nulls.
- `first`, `last` and the element order of `collect_list` depend on how rows arrive, which can change between runs.
  Use `max_by` / `min_by`, or a window with an explicit order, to pick a specific row.

### Example

Profile all orders in one pass, then summarize each customer.

In [ ]:
orders.agg(
    F.count("*").alias("rows"),
    F.count_distinct("customer_id").alias("customers"),
    F.sum("amount").alias("revenue"),
    F.round(F.avg("amount"), 2).alias("avg"),
    F.round(F.mean("amount"), 2).alias("mean"),
    F.min("amount").alias("min"),
    F.max("amount").alias("max"),
    F.median("amount").alias("median"),
).show()

orders.groupBy("customer_id").agg(
    F.count_if(F.col("status") == "paid").alias("paid"),
    F.max_by("order_id", "amount").alias("biggest_order"),
    F.min_by("order_id", "order_ts").alias("first_order"),
    F.sort_array(F.collect_list("status")).alias("all_statuses"),
    F.size(F.collect_set("status")).alias("distinct_statuses"),
    F.first("status").alias("first_seen"),
    F.last("status").alias("last_seen"),
).orderBy("customer_id").show(truncate=False)

**What to notice:** The median (108.20) sits far below the average (221.59) because order 9's 1,180.00 pulls the
average up; report both when a few large values dominate. `max_by` returned the id of each customer's largest
order and `min_by` the earliest, with no join or window. `collect_list` was wrapped in `sort_array` so the output
is stable; `first_seen` and `last_seen` happen to look sensible here, but nothing guarantees which row they pick.

### Distinct sums, variance and correlation

A payment retried by the source appears twice. `sum_distinct` looks like a fix, but it removes repeated *values*,
not repeated *rows*. The statistical aggregates measure spread and how closely two columns move together.

In [ ]:
payments = spark.createDataFrame([(1, 50), (1, 50), (2, 30), (3, 30)], "payment_id int, amount int")
payments.agg(
    F.sum("amount").alias("sum"),
    F.sum_distinct("amount").alias("sum_distinct"),
    F.count("payment_id").alias("count"),
    F.count_distinct("payment_id").alias("count_distinct"),
).show()
payments.dropDuplicates(["payment_id"]).agg(F.sum("amount").alias("deduplicated_sum")).show()

weather = spark.createDataFrame([(18, 120), (21, 135), (25, 160), (28, 190), (31, 205)], "temp_c int, ice_creams int")
weather.agg(
    F.round(F.stddev("ice_creams"), 2).alias("stddev"),
    F.round(F.variance("ice_creams"), 2).alias("variance"),
    F.round(F.corr("temp_c", "ice_creams"), 3).alias("corr"),
).show()

**What to notice:** `sum_distinct` returned 80, not the correct 110: it dropped the retried 50 but also counted 30
once, though payments 2 and 3 are different payments. Deduplicating by key first gives 110. `variance` is the
square of `stddev`, and a correlation close to 1 says sales rise almost in a straight line with temperature.

## Window functions: a preview

A window function computes a value for every row from a group of related rows, without collapsing them. Define
the group with `Window.partitionBy(...).orderBy(...)` and attach the function with `.over(window)`.

| function | returns for each row |
| --- | --- |
| `row_number()` | 1, 2, 3, ... in window order |
| `rank()`, `dense_rank()` | its position, with ties sharing a rank (`rank` leaves gaps after ties) |
| `lag(col, n)`, `lead(col, n)` | the value `n` rows before or after |
| `ntile(n)` | which of `n` roughly equal buckets the row falls in |
| `percent_rank()`, `cume_dist()` | its relative position, from 0 to 1 |

Frames, running totals, moving averages, top-N, deduplication, sessions and gaps-and-islands are on their own
page: [PySpark window functions](pyspark-window-functions.ipynb).

**Where it matters**
- **Ranking and top-N:** each customer's largest order.
- **Change over time:** how this order compares with the previous one.

**Watch out for**
- A window without `partitionBy` puts every row in one partition, processed by a single task.

### Example

Rank customer 1's and 2's orders by amount, and look at the previous and next order in time.

In [ ]:
by_amount = Window.partitionBy("customer_id").orderBy(F.desc("amount"))
by_time = Window.partitionBy("customer_id").orderBy("order_ts")
orders.where("customer_id IN (1, 2)").select(
    "customer_id", "order_id", "amount",
    F.row_number().over(by_amount).alias("row_number"),
    F.rank().over(by_amount).alias("rank"),
    F.dense_rank().over(by_amount).alias("dense_rank"),
    F.ntile(2).over(by_amount).alias("ntile_2"),
    F.round(F.percent_rank().over(by_amount), 2).alias("pct_rank"),
    F.round(F.cume_dist().over(by_amount), 2).alias("cume_dist"),
    F.lag("amount").over(by_time).alias("prev_amount"),
    F.lead("amount").over(by_time).alias("next_amount"),
).orderBy("customer_id", "row_number").show()

**What to notice:** Every original order row is still there; nothing was grouped away. Customer 1's 120.00 order
ranks first by amount with a `pct_rank` of 0 and a `cume_dist` of 0.33 (one of three rows is at or above it).
`lag` and `lead` use the time window, so they follow the order sequence rather than the amount ranking: the
120.00 order is also the first in time, so it has no previous amount.

## Generate ids and hashes

- `sha2(col, 256)` and `md5` produce stable hashes of a value: the same input always gives the same output.
  `xxhash64` is a fast 64-bit hash and `hash` a 32-bit one; both accept several columns. For a composite key, define a stable encoding that preserves field boundaries and null positions before hashing.
- `monotonically_increasing_id()` gives each row a unique 64-bit number, but **not** consecutive or stable:
  the values encode the partition, and change if the data is repartitioned.
- `F.expr("uuid()")` generates a random identifier per row in Spark 4.0. Treat it as non-reproducible across runs.

**Where it matters**
- **Surrogate keys and change detection:** hash the business key, or hash all attributes to spot changed rows.
- **Matching records** within a controlled pipeline when the same canonical input should produce the same key.

**Watch out for**
- Never use `monotonically_increasing_id` or `uuid` as a key you need to reproduce on a rerun.
- Plain concatenation turns both `('ab', 'c')` and `('a', 'bc')` into the same text. A separator can also be ambiguous if it appears in the data.

A plain hash of an email can be guessed by hashing candidate addresses. Treat it as sensitive. Also check ambiguous encodings before hashing a composite key. For example, `concat_ws` with a pipe produces the same text for `("A", null, "B")` and `("A", "B", null)`.

### Example

Hash emails and a row's attributes, and see what `monotonically_increasing_id` produces across partitions.

In [ ]:
customers.select(
    "customer_id",
    F.sha2(F.lower("name"), 256).substr(1, 16).alias("name_sha256"),
    F.hash("customer_id", "name").alias("hash_32bit"),
    F.xxhash64("customer_id", "name", "region").alias("row_hash"),
).show(3, truncate=False)

spark.range(6, numPartitions=3).select("id", F.monotonically_increasing_id().alias("mono_id")).show()

**What to notice:** The hashes are deterministic: rerun the cell and they stay the same. `hash` returns a 32-bit
integer, so across millions of rows it collides far sooner than the 64-bit `xxhash64`; prefer `xxhash64` or `sha2`
for keys. The `mono_id` values jump between partitions (0, 1, then 8589934592, ...) because the upper bits hold
the partition number. They are unique but not consecutive, and a different partitioning would give different
numbers.

### Compare stable hashes and random identifiers

Hash a fixed string and check the shape of a SQL-generated UUID. Check its format because the generated value varies.

In [ ]:
key_text = F.lit("abc")
spark.range(1).select(
    F.sha2(key_text, 256).alias("sha256"),
    F.md5(key_text).alias("md5"),
    F.xxhash64(key_text).alias("xxhash64"),
    F.length(F.expr("uuid()")).alias("uuid_length"),
).show(truncate=False)

**Expected output**

```text
+----------------------------------------------------------------+--------------------------------+-------------------+-----------+
|sha256                                                          |md5                             |xxhash64           |uuid_length|
+----------------------------------------------------------------+--------------------------------+-------------------+-----------+
|ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad|900150983cd24fb0d6963f7d28e17f72|1423657621850124518|36         |
+----------------------------------------------------------------+--------------------------------+-------------------+-----------+
```

**What to notice:** The hashes repeat for this input. The UUID is a 36-character string. Its length is deterministic even though its value is random.

## Use small arrays and maps in column expressions

Collections are useful for fixed mappings, validation messages, and lists of simple values. The [nested-data lesson](pyspark-08-nested-and-semi-structured-data.ipynb) covers working with nested schemas in depth.

| Need | Function | What to watch |
| --- | --- | --- |
| Build a list of flags or messages | `array`, `array_compact` | `array_compact` removes null elements and preserves empty strings |
| Count and test elements | `size`, `array_contains` | `size` counts null elements. `array_contains` can yield null when a null element makes the result unknown |
| Clean or keep selected elements | `transform`, `filter` | Lambdas build Spark column expressions. Use functions such as `F.trim(x)` inside them |
| Test a condition across a list | `exists`, `forall` | `exists` is false and `forall` is true for an empty array. Null elements may give unknown results |
| Deduplicate or sort a list | `array_distinct`, `array_sort` | `array_distinct` keeps one null. Default `array_sort` places nulls last |
| Read an array element safely | `get`, `try_element_at` | `get` is zero-based. `try_element_at` is one-based, supports negative indices from the end, and rejects index zero |
| Map a short vocabulary | `create_map`, `element_at`, `map_contains_key` | A missing key and a present key with a null value both look null when read. Use the key check to distinguish them |
| Join, sort, flatten or pair lists | `array_join`, `sort_array`, `flatten`, `arrays_zip` | `sort_array` keeps duplicates. `arrays_zip` pairs elements by position and pads the shorter array with nulls |
| Reduce a list to one value | `aggregate(arr, start, merge)` | The start value sets the result type, so start a decimal sum with a decimal zero |
| Group fields or read a map | `struct`, `map_keys`, `map_values` | Read a struct field with a dot, as in `profile.name` |
| Turn elements into rows | `explode`, `explode_outer`, `posexplode` | The outer form emits a null row for both a null array and an empty array. Either function can multiply the number of rows |

### Clean, test, and index array elements

Use an array with a duplicate, a null, and an empty string, as well as an empty array.

In [ ]:
array_inputs = spark.createDataFrame(
    [(1, [" b ", None, "a", "a", ""]), (2, [])], "id int, tags array<string>",
)
tag_demo = array_inputs.select("id", F.transform("tags", lambda tag: F.trim(tag)).alias("tags"))
tag_demo.select(
    "id", F.size("tags").alias("size"),
    F.array_contains("tags", "z").alias("has_z"),
    F.array_compact("tags").alias("non_null"),
    F.filter("tags", lambda tag: tag.isNotNull() & (tag != "")).alias("usable"),
    F.exists("tags", lambda tag: tag == "a").alias("any_a"),
    F.forall("tags", lambda tag: tag != "").alias("all_nonempty"),
).orderBy("id").show(truncate=False)
tag_demo.select(
    "id", F.array_distinct("tags").alias("distinct"),
    F.array_sort("tags").alias("sorted"),
    F.get("tags", 0).alias("index_0"),
    F.try_element_at("tags", F.lit(1)).alias("index_1"),
    F.try_element_at("tags", F.lit(99)).alias("past_end"),
).orderBy("id").show(truncate=False)

**Expected output**

```text
+---+----+-----+-----------+---------+-----+------------+
|id |size|has_z|non_null   |usable   |any_a|all_nonempty|
+---+----+-----+-----------+---------+-----+------------+
|1  |5   |NULL |[b, a, a, ]|[b, a, a]|true |false       |
|2  |0   |false|[]         |[]       |false|true        |
+---+----+-----+-----------+---------+-----+------------+

+---+--------------+-----------------+-------+-------+--------+
|id |distinct      |sorted           |index_0|index_1|past_end|
+---+--------------+-----------------+-------+-------+--------+
|1  |[b, NULL, a, ]|[, a, a, b, NULL]|b      |b      |NULL    |
|2  |[]            |[]               |NULL   |NULL   |NULL    |
+---+--------------+-----------------+-------+-------+--------+
```

**What to notice:** A missing match plus a null element makes `has_z` unknown for row 1. Empty arrays make `exists` false and `forall` true. Both indexing functions return the first element with the shown indices and yield null for the empty array. Blank positions in the printed arrays represent empty strings.

### Distinguish absent map keys and expand arrays into rows

Look up statuses in a small map with one explicit null value. Then compare ordinary and outer expansion of arrays.

In [ ]:
stages = F.create_map(F.lit("paid"), F.lit("open"), F.lit("held"), F.lit(None).cast("string"))
status_inputs = spark.createDataFrame([(1, "paid"), (2, "held"), (3, "other")], "id int, status string")
status_inputs.select(
    "id", "status",
    F.element_at(stages, F.col("status")).alias("stage"),
    F.map_contains_key(stages, F.col("status")).alias("key_exists"),
).orderBy("id").show()

expansion = spark.createDataFrame(
    [(1, ["b", "a"]), (2, []), (3, None)], "id int, tags array<string>",
)
expansion.select("id", F.explode("tags").alias("tag")).orderBy("id", "tag").show()
expansion.select("id", F.explode_outer("tags").alias("tag")).orderBy("id", "tag").show()

**Expected output**

```text
+---+------+-----+----------+
| id|status|stage|key_exists|
+---+------+-----+----------+
|  1|  paid| open|      true|
|  2|  held| NULL|      true|
|  3| other| NULL|     false|
+---+------+-----+----------+

+---+---+
| id|tag|
+---+---+
|  1|  a|
|  1|  b|
+---+---+

+---+----+
| id| tag|
+---+----+
|  1|   a|
|  1|   b|
|  2|NULL|
|  3|NULL|
+---+----+
```

**What to notice:** Only `held` is present among the two null lookups. Ordinary expansion emits no rows for IDs 2 and 3. Outer expansion retains each with a null tag.

### Reshape arrays, number elements, and build structs and maps

Two baskets hold parallel `items` and `qty` arrays; the third is empty. Pair, reduce, join and number the elements,
then pack customer fields into a struct and a map.

In [ ]:
baskets = spark.createDataFrame(
    [(1, ["milk", "apple", "milk"], [1, 3, 2]), (2, ["bread"], [5]), (3, [], [])],
    "order_id int, items array<string>, qty array<int>",
)
baskets.select(
    "order_id",
    F.array_join("items", ", ").alias("joined"),
    F.sort_array("items").alias("sorted"),
    F.arrays_zip("items", "qty").alias("lines"),
    F.aggregate("qty", F.lit(0), lambda total, q: total + q).alias("total_qty"),
).orderBy("order_id").show(truncate=False)

baskets.select("order_id", F.posexplode("items").alias("position", "item")).show()

spark.range(1).select(
    F.flatten(F.array(F.array(F.lit(1), F.lit(2)), F.array(F.lit(3)))).alias("flatten")
).show()

profiles = customers.select(
    "customer_id",
    F.struct("name", "region").alias("profile"),
    F.create_map(F.lit("region"), F.col("region"), F.lit("signup"), F.col("signup_date").cast("string")).alias("attrs"),
)
profiles.where("customer_id IN (1, 4)").select(
    "customer_id",
    F.col("profile.name").alias("name"),
    F.map_keys("attrs").alias("keys"),
    F.map_values("attrs").alias("values"),
).show(truncate=False)

**What to notice:** `arrays_zip` turned two parallel arrays into one array of `{items, qty}` pairs, which is safe to
explode in one step. `aggregate` summed basket 1 to 6 and returned the start value 0 for the empty basket.
`posexplode` numbered the positions from 0 and, like `explode`, dropped the empty basket. Customer 4's map still
has a `region` key with a null value, which `map_keys` shows and `element_at` alone would hide.

### Sample problem · Report every order validation error

A small import has order 101 with status `paid` and amount 10, order 102 with a missing status and amount -2, and order 103 with status `cancelled` and a missing amount. A valid status is `placed`, `paid`, `shipped`, or `returned`. Amounts must be present and nonnegative. Return an array of all applicable errors and a boolean `is_valid` flag per order.

In [ ]:
candidate_orders = spark.createDataFrame(
    [(101, "paid", 10), (102, None, -2), (103, "cancelled", None)],
    "order_id int, status string, amount int",
)
status, amount = F.col("status"), F.col("amount")
order_errors = F.array_compact(F.array(
    F.when(status.isNull(), "missing status")
     .when(~status.isin("placed", "paid", "shipped", "returned"), "unsupported status"),
    F.when(amount.isNull(), "missing amount").when(amount < 0, "negative amount"),
))
candidate_orders.withColumn("errors", order_errors).withColumn(
    "is_valid", F.size("errors") == 0,
).orderBy("order_id").show(truncate=False)

**Expected result:** Order 101 has an empty array and is valid. Order 102 has `["missing status", "negative amount"]`. Order 103 has `["unsupported status", "missing amount"]`. Each field contributes its own message, so one error cannot hide another.

## Parse and build JSON

Raw events and API responses often land as JSON text in a string column.

| function | does |
| --- | --- |
| `schema_of_json(sample)` | infers a schema string from one example, a quick start for writing the real schema |
| `from_json(col, schema)` | parses the text into a struct you query with dots; malformed text gives nulls, not an error |
| `get_json_object(col, "$.path")` | pulls one value out as a string, without a schema |
| `json_tuple(col, k1, k2, ...)` | pulls several top-level keys at once, as strings |
| `to_json(col)` | turns a struct, array or map back into JSON text, for example for a Kafka message |

**Where it matters**
- **Bronze to silver:** typing event payloads before they are joined or aggregated.
- **Outgoing messages:** one JSON document per row for a queue or an API.

**Watch out for**
- Parse once with `from_json` and reuse the struct; each `get_json_object` call parses the text again.
- A schema that does not match the data gives nulls rather than errors, so count nulls after parsing.

### Example

Infer a schema, parse three payloads (one is not JSON), extract fields without a schema, and write JSON back out.

In [ ]:
payloads = spark.createDataFrame(
    [(1, '{"device": "ios", "geo": {"country": "DE"}, "tags": ["new", "promo"]}'),
     (2, '{"device": "web", "geo": {"country": "US"}}'),
     (3, "not json")],
    "event_id int, body string",
)
sample = payloads.first().body
print(spark.range(1).select(F.schema_of_json(F.lit(sample))).first()[0])

schema = "device string, geo struct<country string>, tags array<string>"
parsed = payloads.select("event_id", F.from_json("body", schema).alias("data"))
parsed.select("event_id", "data.device", "data.geo.country", "data.tags").orderBy("event_id").show()

payloads.select(
    "event_id",
    F.get_json_object("body", "$.geo.country").alias("country"),
    F.json_tuple("body", "device", "tags").alias("device", "tags_text"),
).orderBy("event_id").show(truncate=False)

parsed.where("event_id = 1").select(F.to_json("data").alias("json")).show(truncate=False)

**What to notice:** `schema_of_json` printed a `STRUCT<...>` type you can adapt for `from_json`. Event 2 has no
`tags`, so that field is null; event 3 is not JSON, so every field is null and the job still succeeds.
`get_json_object` and `json_tuple` return strings, including the raw `["new","promo"]` text for tags, so they
lose the array type. `to_json` rebuilt a compact JSON document from the parsed struct.

## File lineage, partitions and UDFs

- `input_file_name()` returns the file each row was read from. Record it in bronze tables to trace a bad row back
  to its source file.
- `spark_partition_id()` returns the partition a row is in, useful when checking how data is spread.
- `udf(f, returnType)` wraps a Python function that runs once per row; `pandas_udf` runs it on batches of rows as
  pandas Series through Arrow, which is usually much faster. [Lesson 9](pyspark-09-python-udfs-and-pandas.ipynb)
  covers both.

**Where it matters**
- **Auditing loads:** which file did this row come from?
- **Logic Spark lacks,** such as a call into a specialist Python library.

**Watch out for**
- A UDF is a black box: Spark cannot optimize through it and must ship every row to a Python worker. Look for a
  built-in function first; most string, date and array work has one.
- `input_file_name` is empty for data that did not come from files.

### Example

Write orders as two CSV files and read them back with the source file and partition. Then mask emails with a
Python UDF and add VAT with a pandas UDF.

In [ ]:
import pandas as pd

drop_dir = os.path.join(WORK_DIR, "drops")
orders.select("order_id", "amount").repartition(2).write.mode("overwrite").csv(drop_dir, header=True)
landed = spark.read.csv(drop_dir, header=True, schema="order_id int, amount decimal(12,2)")
landed.select(
    "order_id",
    F.substring(F.element_at(F.split(F.input_file_name(), "/"), -1), 1, 10).alias("source_file"),
    F.spark_partition_id().alias("partition"),
).orderBy("order_id").show(4)


@F.udf("string")
def mask_email(email):
    if email is None:
        return None
    user, _, domain = email.partition("@")
    return user[0] + "***@" + domain


@F.pandas_udf("double")
def with_vat(amount: pd.Series) -> pd.Series:
    return amount.astype(float) * 1.2


people.select("email", mask_email(F.lower(F.trim("email"))).alias("masked")).show(truncate=False)
orders.select("order_id", "amount", with_vat("amount").alias("gross")).show(3)

**What to notice:** Every row carries the `part-0000...` file it was read from and its partition, so a bad value
can be traced to one file. `mask_email` ran Python once per row; `with_vat` received whole pandas Series. The VAT
is also a one-liner with built-ins (`F.col("amount") * 1.2`), which runs faster and keeps the `decimal` type, so
in real code the pandas UDF would be the wrong choice.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why store money as `decimal` rather than `double`?

**A.** A `double` stores binary approximations, so sums drift (ten times 0.1 is 0.9999999999999999). A `decimal` is exact.

**Q2.** In Spark 4, what does casting `"N/A"` to `int` do, and how do you get a null instead?

**A.** ANSI mode is on, so the cast raises `CAST_INVALID_INPUT` and the job fails. `try_cast` returns null.

**Q3.** Why does `region == None` never select anything?

**A.** Any comparison with null gives null, and a filter keeps only true rows. Use `isNull()` or `eqNullSafe`.

**Q4.** What is the difference between `count('*')` and `count('region')`?

**A.** `count('*')` counts rows; `count('region')` counts non-null values of `region`.

**Q5.** Why might `concat` return null where `concat_ws` does not?

**A.** `concat` returns null if any input is null; `concat_ws` skips nulls.

**Q6.** Can you use `monotonically_increasing_id` as a primary key?

**A.** Only within one run. The values depend on partitioning, are not consecutive, and change on a rerun. Hash a business key instead.

**Q7.** Where do nulls go in `orderBy(F.asc("region"))`, and how do you move them to the end?

**A.** First, because ascending order puts nulls first. Use `F.asc_nulls_last("region")`.

**Q8.** How do you get the id of each customer's largest order without a join or a window?

**A.** `F.max_by("order_id", "amount")` inside `groupBy("customer_id").agg(...)`.

**Q9.** A retried payment appears twice. Why is `sum_distinct("amount")` the wrong fix?

**A.** It removes repeated amounts, not repeated payments, so two different payments of the same amount count once. Deduplicate by payment id, then `sum`.

**Q10.** Why prefer `from_json` over several `get_json_object` calls on the same column?

**A.** `from_json` parses the text once into a typed struct; every `get_json_object` call parses it again and returns only strings.

**Q11.** When is a UDF the right choice?

**A.** Only when no built-in function or combination of them can express the logic. Built-ins are optimized by Spark and avoid moving rows to Python.

## Key takeaways

- Money is `decimal`; measurements can be `double`.
- Spark 4 casts fail loudly; use `try_` functions where bad values are expected, then count the nulls.
- Nulls fail comparisons and are skipped by aggregates: handle them on purpose.
- Order `when` branches from the most specific; always decide what `otherwise` should be.
- Give date formats explicitly and pin the session time zone.
- Hash business keys for reproducible ids.
- Say where nulls sort with `asc_nulls_last` and friends whenever order matters.
- Pick specific rows with `max_by`, `min_by` or a window, never with `first` or `last`.
- Parse JSON once with `from_json`; reach for a UDF only when no built-in function fits.

**Next:** Practise this module in [PySpark drills 3: types, functions and nulls](pyspark-drills-03-types-functions-and-nulls.ipynb), then continue with [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb).